# Epic versus tragedy

In [1]:
import numpy as np
import pandas as pd

homer_df = pd.read_parquet("../parquet/homer_dtm.parquet")
tragedy_df = pd.read_parquet("../parquet/tragedy_dtm.parquet")

In [4]:
def log_likelihood(a: pd.Series, b: pd.Series, total_a: int, total_b: int) -> pd.Series:
    """Dunning G² for each lemma. Positive = skewed toward corpus A."""
    N = total_a + total_b
    expected_a = (a + b) * total_a / N
    expected_b = (a + b) * total_b / N

    def safe_term(obs, exp):
        return np.where(obs > 0, obs * np.log(obs / exp), 0)

    g2 = 2 * (safe_term(a, expected_a) + safe_term(b, expected_b))
    return pd.Series(np.where(a / total_a >= b / total_b, g2, -g2), index=a.index)

In [17]:
il = homer_df["iliad"]
od = homer_df["odyssey"]
homer_series = il + od
tragedy_series = tragedy_df.sum(axis=1)

homer_series, tragedy_series = homer_series.align(tragedy_series, fill_value=0)

total_homer = int(homer_series.sum())
total_tragedy = int(tragedy_series.sum())

loglikelihood_epic_tragedy = log_likelihood(homer_series, tragedy_series, total_homer, total_tragedy)


/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [31]:
import altair as alt

from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

total_tokens = total_homer + total_tragedy
overall_freq = (homer_series + tragedy_series) / total_tokens

epic_tragedy_plot_df = pd.DataFrame({
    "lemma": loglikelihood_epic_tragedy.index,
    "log_likelihood": loglikelihood_epic_tragedy.values,
    "relative_frequency": overall_freq.values,
    "register": np.where(loglikelihood_epic_tragedy.values >= 0, "Epic", "Tragic"),
})

# p-value from χ^2(df=1); use abs() because G^2 is signed here
epic_tragedy_plot_df["p_value"] = chi2.sf(epic_tragedy_plot_df["log_likelihood"].abs(), df=1)

# Benjamini-Hochberg FDR correction across all lemmata
_, epic_tragedy_plot_df["p_value_fdr"], _, _ = multipletests(epic_tragedy_plot_df["p_value"], method="fdr_bh")

epic_tragedy_plot_df = epic_tragedy_plot_df[epic_tragedy_plot_df["log_likelihood"].abs() >= 30]
print(f"{len(epic_tragedy_plot_df)} lemmata after filtering")
epic_tragedy_plot_df.sort_values("log_likelihood", ascending=False).tail(10)

949 lemmata after filtering


,lemma,log_likelihood,relative_frequency,register,p_value,p_value_fdr
18708,τέκνον,-400.318045,0.003024,Tragic,4.695699e-89,6.910639e-86
19714,τόδις,-407.019517,0.001439,Tragic,1.632701e-90,2.631681e-87
12952,οὗς,-433.929899,0.001534,Tragic,2.267540e-96,4.039682e-93
15230,ποός,-436.172430,0.001542,Tragic,7.370231e-97,1.385972e-93
3554,γῆ,-628.983480,0.002596,Tragic,8.315827e-139,2.010589e-135
5775,εἷς,-686.264377,0.004281,Tragic,2.901668e-151,8.928959e-148
6992,θνήσκω,-703.033712,0.002485,Tragic,6.546447e-155,2.215907e-151
31460,ἵ,-858.889670,0.003036,Tragic,8.490731e-189,5.748055e-185
10511,λόγος,-895.868695,0.003262,Tragic,7.761025e-197,6.567573e-193
19298,τιξ,-1028.200819,0.003635,Tragic,1.332014e-225,2.254367e-221


In [32]:
alt.Chart(epic_tragedy_plot_df).mark_point(opacity=0.5, size=30).encode(
    x=alt.X("log_likelihood:Q", title="Log-likelihood (← Tragic | Epic →)"),
    y=alt.Y(
        "relative_frequency:Q",
        title="Overall relative frequency",
        scale=alt.Scale(type="log"),
    ),
    color=alt.Color(
        "register:N",
        scale=alt.Scale(domain=["Epic", "Tragic"], range=["tomato", "steelblue"]),
        legend=alt.Legend(title="More common in"),
    ),
    tooltip=["lemma:N", "log_likelihood:Q", "relative_frequency:Q"],
).properties(
    title="Epic vs. Tragic: lemma distinctiveness vs. overall frequency",
    width=650,
    height=450,
)

alt.Chart(...)

In [30]:
epic_tragedy_plot_df.sort_values("log_likelihood", ascending=False).to_csv("../csv/epic_tragic.csv")